# Instalação das dependências

In [ ]:
import numpy as np
import segyio
import matplotlib.pyplot as plt
import cv2
import pickle

import os
import sys
sys.path.append('..')

from scipy.ndimage import zoom
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.metrics import jaccard_score, f1_score, accuracy_score, recall_score
from sklearn.cluster import KMeans
from tqdm import tqdm

from atributos import Atributos
from superficie import Superficie
from utils import plot_Fernanda, plot_VJ, concatenate, subplot, segmenta, colore, resample
from hparams import drive_fernanda, drive_vittor

# Leitura dos arquivos CSV das superfícies
isso aqui é pra ler os horizontes q é a label dessa segmentacao, a classe Superficie vai contar a tabela de todas as seçoes anotadas e tem a funçao pra carregar a classe Atributos com a label daquela seçao especifica

In [ ]:
barravelha = Superficie(os.path.join(drive_vittor, 'Horizontes/Topo_BV.csv'), shape = [3400, 3000], offsets = [27700, 45400])
camboriu = Superficie(os.path.join(drive_vittor, 'Horizontes/Topo_Embasamento.csv'), shape = [3400, 3000], offsets = [27700, 45400])
barravelha_Fernanda = Superficie(os.path.join(drive_fernanda, 'Horizontes/Topo_BV_Fernanda.csv'), shape = [1950, 1710], offsets = [2100, 3285])
camboriu_Fernanda = Superficie(os.path.join(drive_fernanda, 'Horizontes/Topo_Embasamento_Fernanda.csv'), shape = [1950, 1710], offsets = [2100, 3285])

# Crosslines/Inlines selecionadas para predição

Evite adicionar crosslines/inlines de treino na predição. Fundamentalmente não faz sentido, e evita o resample e segmentação nas etapas seguintes, o que talvez cause algum resultado inesperado devido a etapa de crop da velocidade durante a reamostragem.

Essas listas abaixos sao separadas apenas para que no final dê pra fazer os plots das predições separadas

In [ ]:
sessoes_predicao_Vittor = [
    # Atributos(
    #     path_sismica = os.path.join(drive_vittor, 'IL_46900.sgy'),
    #     path_velocidade = os.path.join(drive_vittor, 'IL_46900_vel.sgy'),
    #     resolucao = 6.25,
    #     deltaz = [12000, 0],
    #     linha_barravelha = barravelha.line(i = 46900, inline = 0, offset = False),
    #     linha_camboriu = camboriu.line(i = 46900, inline = 0, offset = False)
    # ),
]
sessoes_predicao_Fernanda = [
    # Atributos(
    #     path_sismica = os.path.join(drive_fernanda, 'CL_3300_Fernanda.sgy'),
    #     path_velocidade = os.path.join(drive_fernanda, 'CL_3300_Vel_Fernanda.sgy'),
    #     resolucao = 25,
    #     deltaz = [9955, 2000],
    #     linha_barravelha = barravelha_Fernanda.line(i = 3300 - 2100, inline = 1, offset = True),
    #     linha_camboriu = camboriu_Fernanda.line(i = 3300 - 2100, inline = 1, offset = True)
    # ),
]

# atributos

In [ ]:
for obj in tqdm(sessoes_predicao_Vittor):
    obj._complexo()
    obj._logaritmo()
    obj._coherence()
    obj._soterramento(marambaia = False) # soterramento é o yy
    obj._tecva()

for obj in tqdm(sessoes_predicao_Fernanda):
    obj._complexo()
    obj._logaritmo()
    obj._coherence()
    obj._soterramento(marambaia = False) # soterramento é o yy
    obj._tecva()

# Utilizando o modelo

In [ ]:
def mistura(obj):
    return [
        obj.X, 
        obj.envelope, obj.fase, obj.freq, obj.sweetness, # complexo
        obj.log, # logaritmo
        obj.gersztenkorn, obj.sobel, obj.marfurt, # coherence
        obj.yy, # soterramento
        obj.tecva, obj.rms, # tecva
        obj.velocidade_rs, # velocidade
    ]

def compute_metrics(y, yhat):
    jaccard = jaccard_score(y, yhat)
    dice = f1_score(y, yhat)
    accuracy = accuracy_score(y, yhat)
    recall = recall_score(y, yhat)
    print('jaccard {}, dice {}, acc {}, recall {}'.format(jaccard, dice, accuracy, recall))

with open('./modelo.pkl', 'rb') as inp: 
    clf = pickle.load(inp)

for obj in sessoes_predicao_Vittor:
    obj.yhat = clf.predict(concatenate(mistura(obj))).reshape(obj.X.shape)
    plot_VJ(obj, obj.yhat, color = 'tab10', linhas = True)
    compute_metrics(obj.segmentado.ravel(), obj.yhat.ravel())
for obj in sessoes_predicao_Fernanda:
    obj.yhat = clf.predict(concatenate(mistura(obj))).reshape(obj.X.shape)
    plot_Fernanda(obj, obj.yhat, color = 'tab10', linhas = True)
    compute_metrics(obj.segmentado.ravel(), obj.yhat.ravel())
